# 🏥 ViBioMIR 2026 — Dual-GPU Fast Inference Runner (Macro F2 Optimized)

> **Thiết lập Kaggle quan trọng trước khi bấm Run All:**
> 1. **Accelerator**: Chọn **GPU T4 x 2** (Dual GPU để reranker chạy song song 2 GPU siêu tốc).
> 2. **Internet**: Bật **ON** (để tải thư viện và index).
> 3. **Secrets**: Vào menu **Add-ons -> Secrets**, thêm secret tên `HF_TOKEN` (token Hugging Face có quyền Read) và **tích chọn checkbox** bên cạnh.

⚡ **Điểm nổi bật của phiên bản này:**
- **Đồng bộ 100% BM25s + Dense FAISS SQ8**: Toàn bộ 2,318,540 chunks (750 ký tự chuẩn LCS >= 0.40).
- **Siêu tốc (~35–45 phút)**: Tối ưu `--hybrid-top-k 200`, `--expand-top-docs 0` và offline PubMed cache thay vì 3 tiếng.
- **Macro F2 Sweet Spot**: Cân bằng Recall (trọng số 4x) và Precision với `--top-k-docs 60`, `--top-k-chunks 100`, `--max-chunks-per-doc 2`.


In [ ]:
!nvidia-smi    

In [ ]:
# Cấu hình hiển thị chống tràn dòng chữ ngang trên Kaggle
from IPython.display import HTML, display
display(HTML("<style>pre { white-space: pre-wrap !important; word-break: break-word !important; }</style>"))
print("✓ Đã kích hoạt chế độ hiển thị tự động xuống dòng trên Kaggle!")


In [ ]:
!pip install -q uv

In [ ]:
!uv pip install --system --no-cache faiss-gpu pyarrow "huggingface_hub[cli]" bm25s rank-bm25 pyvi jieba loguru typer pydantic pyyaml tqdm flagembedding sentence-transformers trafilatura httpx qdrant_client

In [ ]:
import os                                                                                                                                                                             
from pathlib import Path                                                                                                                                                              
                                                                                                                                                                                      
REPO_URL = "https://github.com/neiths/med-doc-retrieval.git"                                                                                                                          
PROJECT_DIR = Path("/kaggle/working/med-doc-retrieval")                                                                                                                               
                                                                                                                                                                                      
if not PROJECT_DIR.exists():                                                                                                                                                          
    print("Cloning repository...")                                                                                                                                                    
    !git clone -b main $REPO_URL $PROJECT_DIR                                                                                                                                         
else:                                                                                                                                                                                 
    print("Updating repository...")                                                                                                                                                   
    %cd $PROJECT_DIR                                                                                                                                                                  
    !git pull origin main                                                                                                                                                             
                                                                                                                                                                                      
%cd $PROJECT_DIR                                                                                                                                                                      
print("Current Working Directory:", os.getcwd()) 

In [ ]:
import os
import glob
import json
from pathlib import Path
from huggingface_hub import HfApi, login

HF_TOKEN = ""
try:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
except Exception:
    pass

if not HF_TOKEN:
    HF_TOKEN = os.environ.get("HF_TOKEN", "")

if not HF_TOKEN or not HF_TOKEN.strip():
    print("⚠️ Hãy vào menu Add-ons -> Secrets trên Kaggle và TÍCH CHỌN checkbox cạnh HF_TOKEN.")
    HF_TOKEN = input("Hoặc nhập token vào đây: ").strip()

os.environ["HF_TOKEN"] = HF_TOKEN
login(token=HF_TOKEN)
api = HfApi(token=HF_TOKEN)

# 1. Tạo sẵn thư mục lưu trữ
os.makedirs("data/indices/bm25s_index", exist_ok=True)
os.makedirs("data/processed", exist_ok=True)

# 2. Kiểm tra nếu có metadata BM25 cũ (< 2,318,540 chunks) thì dọn dẹp để tải bộ mới
meta_p = Path("data/indices/bm25s_metadata.json")
if meta_p.exists():
    try:
        with open(meta_p, "r", encoding="utf-8") as f:
            meta_chk = json.load(f)
        if meta_chk.get("num_chunks") != 2318540:
            print("⚠️ Phát hiện metadata BM25 cũ. Đang dọn dẹp để tải bộ 2.32M mới nhất...")
            meta_p.unlink(missing_ok=True)
            for f_cand in Path("data/indices/bm25s_index").glob("*"):
                f_cand.unlink(missing_ok=True)
    except Exception:
        pass

# 3. Kiểm tra xem đã có FAISS SQ8 và SQLite Slim trong Kaggle Input chưa
has_kaggle_dense = bool(glob.glob("/kaggle/input/**/*.faiss", recursive=True))
has_kaggle_sqlite = bool(glob.glob("/kaggle/input/**/chunks_meta*.sqlite", recursive=True))

# 4. Danh sách các file cần tải từ HF Bucket (nieths/ViBioMIR)
all_files = [
    ("indices/bm25s_index/data.csc.index.npy", "data/indices/bm25s_index/data.csc.index.npy"),
    ("indices/bm25s_index/indices.csc.index.npy", "data/indices/bm25s_index/indices.csc.index.npy"),
    ("indices/bm25s_index/indptr.csc.index.npy", "data/indices/bm25s_index/indptr.csc.index.npy"),
    ("indices/bm25s_index/params.index.json", "data/indices/bm25s_index/params.index.json"),
    ("indices/bm25s_index/vocab.index.json", "data/indices/bm25s_index/vocab.index.json"),
    ("indices/bm25s_metadata.json", "data/indices/bm25s_metadata.json"),
    ("processed/pubmed_cache.jsonl", "data/processed/pubmed_cache.jsonl"),
    ("processed/queries_enriched.jsonl", "data/processed/queries_enriched.jsonl"),
]

if not has_kaggle_dense:
    all_files.extend([
        ("indices/dense_index_sq8.faiss", "data/indices/dense_index_sq8.faiss"),
        ("indices/dense_metadata_sq8.json", "data/indices/dense_metadata_sq8.json"),
    ])
else:
    print("✓ Đã phát hiện Dense FAISS Index trong Kaggle Input!")

if not has_kaggle_sqlite:
    all_files.append(("indices/chunks_meta_slim.sqlite", "data/indices/chunks_meta_slim.sqlite"))
else:
    print("✓ Đã phát hiện Chunks SQLite trong Kaggle Input!")

# 5. Lọc ra các file chưa có hoặc dung lượng bằng 0
missing_files = []
for remote, local in all_files:
    p = Path(local)
    if not p.exists() or p.stat().st_size == 0:
        missing_files.append((remote, local))
    else:
        print(f"✓ Đã có sẵn: {local} ({p.stat().st_size / (1024*1024):.2f} MB)")

# 6. Tải các file còn thiếu siêu tốc từ HF Bucket
if missing_files:
    print(f"
📥 Đang tải {len(missing_files)} file còn thiếu từ HF Bucket nieths/ViBioMIR...")
    api.download_bucket_files(
        bucket_id="nieths/ViBioMIR",
        files=missing_files,
        token=HF_TOKEN,
    )
    print("✅ Tải xong toàn bộ các file chỉ mục cần thiết!")
else:
    print("
✅ Tất cả các file chỉ mục (Dense SQ8 + BM25s + SQLite Slim + Caches) đã sẵn sàng!")


In [ ]:
!ls -lh data/indices

In [ ]:
# 6. Chạy inference IN-PROCESS — reranker tự động dùng song song cả 2 GPU T4
import glob
import os
import runpy
import sys
from pathlib import Path

os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

import torch
from huggingface_hub import hf_hub_download

dense_arg = "data/indices"
meta_arg = "data/indices/chunks_meta_slim.sqlite"

k_faiss = glob.glob("/kaggle/input/**/*.faiss", recursive=True)
k_sqlite = glob.glob("/kaggle/input/**/chunks_meta*.sqlite", recursive=True)
if k_faiss:
    dense_arg = str(Path(k_faiss[0]).parent)
    print(f"✓ Dense FAISS: {k_faiss[0]}")
if k_sqlite:
    meta_arg = k_sqlite[0]
    print(f"✓ Chunks SQLite: {k_sqlite[0]}")

# Ưu tiên tập truy vấn làm giàu sẵn (queries_enriched.jsonl) để tra cứu PubMed offline 0ms
enriched_path = Path("data/processed/queries_enriched.jsonl")
k_query = glob.glob("/kaggle/input/**/query.parquet", recursive=True)

if enriched_path.exists() and enriched_path.stat().st_size > 0:
    queries_path = enriched_path
    print(f"✓ Sử dụng truy vấn làm giàu sẵn (Offline PubMed Accelerated): {queries_path}")
elif k_query:
    queries_path = Path(k_query[0])
    print(f"✓ Sử dụng query parquet từ Kaggle Input: {queries_path}")
else:
    queries_path = Path("data/raw/vibio_mir/query.parquet")
    if not queries_path.exists():
        queries_path.parent.mkdir(parents=True, exist_ok=True)
        queries_path = Path(
            hf_hub_download(
                repo_id="AIGuruTinix/ViBioMIR",
                filename="query.parquet",
                repo_type="dataset",
                local_dir=str(queries_path.parent),
            )
        )
    print(f"✓ Sử dụng query parquet: {queries_path}")

gpu_count = torch.cuda.device_count()
gpu_names = [torch.cuda.get_device_name(i) for i in range(gpu_count)]
print(f"GPU khả dụng: {gpu_count} -> {gpu_names}")
if gpu_count < 2:
    print("⚠️ Chỉ thấy 1 GPU — Hãy chọn Accelerator 'GPU T4 x2' để đạt tốc độ tối đa.")

os.makedirs("outputs/submissions", exist_ok=True)

# Dùng file checkpoint riêng biệt để không bị dính kết quả partial cũ của lần trước
ckpt_path = "outputs/submissions/run_v2_sweetspot.partial.jsonl"

# Cấu hình Sweet Spot tối ưu để vượt qua 0.1880:
# - hybrid-top-k: 300 (phủ rộng ứng viên từ cả FAISS SQ8 và BM25s mới 2.32M)
# - expand-top-docs: 25 (mở rộng lấy các chunk trong top 25 docs để cứu Chunk Recall)
# - expand-max-chunks-per-doc: 5 (lấy tối đa 5 chunks khớp từ khóa nhất mỗi doc mở rộng)
# - top-k-chunks: 150 (tận dụng trọng số Recall 4x trong F2)
# - top-k-docs: 120 (phủ rộng doc)
# - max-chunks-per-doc: 3 (cho phép 2-3 chunk quan trọng cùng xuất hiện)
# - batch-size: 32 (ổn định VRAM trên Dual T4)
# - disable-pubmed-network: dùng 100% offline cache 0ms
sys.argv = [
    "scripts/run_inference_only.py",
    "--queries", str(queries_path),
    "--dense-dir", dense_arg,
    "--chunks-meta-path", meta_arg,
    "--hybrid-top-k", "300",
    "--top-k-docs", "120",
    "--top-k-chunks", "150",
    "--max-chunks-per-doc", "3",
    "--expand-top-docs", "25",
    "--expand-max-chunks-per-doc", "5",
    "--batch-size", "32",
    "--disable-pubmed-network",
    "--checkpoint", ckpt_path,
]
print("\n🚀 Bắt đầu chạy inference Sweet Spot (dự kiến: ~55-60 phút trên Dual T4)...\n")
runpy.run_path("scripts/run_inference_only.py", run_name="__main__")


In [ ]:
import base64
from IPython.display import HTML, display

zip_path = "/kaggle/working/submission.zip"
with open(zip_path, "rb") as f:
    data_b64 = base64.b64encode(f.read()).decode()

html_button = f'''
<a href="data:application/zip;base64,{data_b64}" download="submission.zip" 
   style="display:inline-block; padding:12px 24px; background:#007acc; color:white; 
          text-decoration:none; border-radius:6px; font-weight:bold; font-size:15px;">
   ⬇ BẤM VÀO ĐÂY ĐỂ TẢI SUBMISSION.ZIP VỀ MÁY
</a>
'''
display(HTML(html_button))

In [ ]:
# 8. Kiểm tra chi tiết cấu trúc file submission.zip vừa tạo
import json
import zipfile
from pathlib import Path

sub_zip = Path("/kaggle/working/submission.zip")
if not sub_zip.exists():
    sub_zip = Path("outputs/submissions/submission.zip")
if not sub_zip.exists():
    sub_zip = Path("submission.zip")

if sub_zip.exists():
    with zipfile.ZipFile(sub_zip, "r") as z:
        print(f"✓ File trong submission.zip: {z.namelist()}")
        with z.open("submission.json") as f:
            sample_data = json.load(f)
            total_q = len(sample_data)
            print(f"✓ Tổng số queries: {total_q:,} (Kỳ vọng: 1,200)")
            
            doc_counts = [len(q.get("relevant_docs", [])) for q in sample_data]
            chunk_counts = [len(q.get("relevant_chunks", [])) for q in sample_data]
            
            avg_docs = sum(doc_counts) / max(total_q, 1)
            avg_chunks = sum(chunk_counts) / max(total_q, 1)
            
            print(f"✓ Số docs trung bình / query:   {avg_docs:.1f} (Min: {min(doc_counts)}, Max: {max(doc_counts)})")
            print(f"✓ Số chunks trung bình / query: {avg_chunks:.1f} (Min: {min(chunk_counts)}, Max: {max(chunk_counts)})")
            
            if sample_data:
                q0 = sample_data[0]
                print(f"
  Mẫu Query ID {q0.get('id')}:")
                print(f"    - Docs dự đoán ({len(q0.get('relevant_docs', []))} docs): {q0.get('relevant_docs', [])[:10]}...")
                if q0.get("relevant_chunks"):
                    c0 = q0["relevant_chunks"][0]
                    chunk_text = c0.get("chunk_text", "")
                    print(f"    - Chunk #1 [Doc {c0.get('doc_id')}]: {len(chunk_text)} ký tự")
                    print(f"      "{chunk_text[:120]}..."")
    print("
✅ File submission hoàn toàn hợp lệ và đã sẵn sàng nộp!")
else:
    print("⚠️ Chưa tìm thấy submission.zip. Hãy chạy Cell 6 trước!")
